# IDA CFG Analysis with ipysigma

This notebook demonstrates how to extract, load, and visualize the Control Flow Graph (CFG) from an IDA database.


### 1. Extract CFG from IDA Database
This step requires `idapro` 9.0+ and a valid license.


In [ ]:
import sys
import os
from src.extract_cfg import extract_cfg_from_db

# Replace with the path to your IDA database (.idb or .i64)
db_path = "/Users/mark/windows_share/test/reorder_and_pad.exe.i64"
json_path = "reorder_and_pad2.json"

if os.path.exists(db_path):
    cfg_dict = extract_cfg_from_db(db_path, output_path=json_path)
else:
    print(f"Database {db_path} not found. Skipping extraction.")


### 2. Load and Visualize the Graph with ipysigma


In [ ]:
#from idadex import idaapi
from typing import Hashable
from networkx import k_core, DiGraph
import importlib
importlib.invalidate_caches()
from ipysigma import Sigma
import pandas as pd
import networkx as nx
import src.visualize_cfg as cfg
importlib.reload(cfg)
import os
import src.hierarchical_summaries as st
from src.graph_viz_layout import function_cluster_layout, add_force_weights
from src.function_call_graph import get_level0_summary_graph
from src.hierarchical_summaries import HierchicalGraphSummarizer


json_path = "reorder_and_pad2.json"

def prune_graph(og: DiGraph[Hashable]) -> DiGraph[Hashable]:
    to_return = og.copy()
    print(f"Graph loaded: {len(to_return.nodes)} nodes, {len(to_return.edges)} edges")
    entrypoint = [x for x in to_return.nodes() if to_return.nodes[x]['entry_point'] == True]
    print(f"entrypoint is {entrypoint}")

    # remove self loops
    to_return.remove_edges_from(nx.selfloop_edges(to_return))

    to_return = collapse_thunks(to_return)
    to_return = cfg.collapse_chains(to_return)

    # remove nodes with degree <= 1 - ALSO REMOVING ISOLATED LEAVES
    #to_return = k_core(to_return, 2)
    #entrypoint = [x for x in to_return.nodes() if to_return.nodes[x]['entry_point'] == True]
    #print(f"entrypoint is {entrypoint}")
    # to_be_removed = [x for  x in G.nodes() if G.degree()[x] <= 1]
    # print(f"Number of nodes to be removed: {len(to_be_removed)}")
    # G.remove_nodes_from(to_be_removed)
    # # Basic info
    # print(f"Number of functions after removing degree <= 1: {len(G.nodes)}")
    #
    # print(f"Number of nodes after collapsing chains: {len(G.nodes)}")
    # entrypoint = [x for x in G.nodes() if G.nodes[x]['entry_point'] == True]
    # if not entrypoint:
    #     print("No entrypoint found. Please check the graph.")
    #     raise ValueError("No entrypoint found")
    print(f"Graph pruned: {len(to_return.nodes)} nodes, {len(to_return.edges)} edges")
    return to_return

def collapse_thunks(G: nx.DiGraph):
    collapsed_G = G.copy()
    nodes_to_process = list(collapsed_G.nodes())

    for node in nodes_to_process:

        flags = collapsed_G.nodes[node].get('flags', {})
        if flags & 128:
            preds = list(collapsed_G.predecessors(node))
            succs = list(collapsed_G.successors(node))
            if len(succs) == 1:
                collapsed_G.remove_node(node)
                for pred in preds:
                    collapsed_G.add_edge(pred, succs[0])
    return collapsed_G

# Load the graph data
G = cfg.load_cfg(json_path)
if G is None:
    print(f"Could not load graph from {json_path}. Please make sure the file exists and it is a valid CFG JSON.")
    exit()

assert(G is not None)
pruned = prune_graph(G)

largest_degree = 0
def set_size(n, d) -> int:
    return len(d.get('instrs'))+G.out_degree(n)

Ways to reduce:
- strongly connected components
- remove nodes with degree <= 1
- clique detection and collapse
- graph community collapse
min cuts
- graph spectral?
- laplacian?
- adamic-adar index?

In [ ]:
## needed to reload code changed on disk, this is the correct import
import src.summarize_graph as sg
importlib.reload(sg)
summaries = sg.summarize_graph(pruned, base_url="http://192.168.1.101:8000/v1", max_concurrent=256, model="qwen3-coder-next", cache_path="./cache_full_func_fix_test.json")


In [ ]:
for node in pruned.nodes():
    pruned.nodes[node]['summary'] = summaries.get(pruned.nodes[node].get('func'))

In [ ]:
from collections import Counter

Counter(d.get("type", "unknown") for _, _, d in pruned.edges(data=True))

# Control-Flow-Graph (Block Level)

In [ ]:
# visualize with ipysigma, now with summaries
weighted_g = add_force_weights(pruned)

layout = function_cluster_layout(weighted_g,radius=12, spacing=120)

widget = Sigma(
    weighted_g,
    layout=layout,
    start_layout=False,
    node_label="label",
    raw_node_color=lambda n, d: d.get("color"),
    edge_color="type",
    default_edge_type="arrow",
    height=800
)
display(widget)

# Function Call-graph 

In [ ]:
level0_summary_graph = get_level0_summary_graph(pruned)

In [ ]:
# Visualise function call graph
widget = Sigma(
    level0_summary_graph,
    start_layout=False,
    node_label="name",
    raw_node_color=lambda n, d: d.get("color"),
    edge_color="type",
    default_edge_type="arrow",
    height=800
)
display(widget)


# Hierarchical Summaries

In [ ]:
summarizer = HierchicalGraphSummarizer(level0_summary_graph)

In [ ]:
await summarizer.execute_pipeline()

# Visualize Hierarchical Summaries

In [ ]:
import json

def short_text(text, limit=240):
    text = " ".join(str(text or "").split())
    return text if len(text) <= limit else text[: limit - 1] + "..."


def build_hierarchical_summary_viz_graph(summary_path="hierarchical_summaries.json", function_graph=None):
    with open(summary_path, "r") as f:
        summaries = json.load(f)

    H = nx.DiGraph()
    function_to_components = {}

    for community_id, community in summaries.items():
        source_ids = community.get("source_ids", [])
        summary = community.get("summary", "")
        has_error = "LLM Generation Error" in summary or "invalid_api_key" in summary
        kind = "error_component" if has_error else "component"
        if community_id == "isolated_func":
            kind = "isolated_component"

        H.add_node(
            community_id,
            name=community.get("title") or community_id,
            label=community.get("title") or community_id,
            kind=kind,
            level=community.get("level", 0),
            summary=summary,
            hover=short_text(summary),
            size=max(8, min(40, 8 + len(source_ids) ** 0.5 * 4)),
            color="#c0392b" if has_error else ("#7f8c8d" if community_id == "isolated_func" else "#2c7fb8"),
            source_count=len(source_ids),
        )

        for detail in community.get("source_details", []):
            func_id = detail.get("id") or detail.get("name")
            if not func_id:
                continue

            function_to_components.setdefault(func_id, set()).add(community_id)
            if func_id not in H:
                H.add_node(
                    func_id,
                    name=detail.get("name", func_id),
                    label=detail.get("name", func_id),
                    kind="function",
                    level=0,
                    summary=detail.get("summary", ""),
                    hover=short_text(detail.get("summary", "")),
                    size=3,
                    color="#95a5a6",
                    source_count=1,
                )

            H.add_edge(
                community_id,
                func_id,
                type="contains",
                edge_type="contains",
                weight=1,
            )

    # If level0_summary_graph exists, add rolled-up edges between summary components.
    if function_graph is not None:
        for src, dst, edge_data in function_graph.edges(data=True):
            src_components = function_to_components.get(src, set())
            dst_components = function_to_components.get(dst, set())
            for src_component in src_components:
                for dst_component in dst_components:
                    if src_component == dst_component:
                        continue
                    if H.has_edge(src_component, dst_component):
                        H.edges[src_component, dst_component]["weight"] += 1
                        H.edges[src_component, dst_component]["call_count"] += edge_data.get("call_count", 1)
                    else:
                        H.add_edge(
                            src_component,
                            dst_component,
                            type="component-call",
                            edge_type="component-call",
                            weight=2,
                            call_count=edge_data.get("call_count", 1),
                        )

    return H, summaries


hierchical_sum_graph, hierarchical_summary_data = build_hierarchical_summary_viz_graph(
    "hierarchical_summaries.json",
    function_graph=globals().get("level0_summary_graph"),
)

print(
    f"Hierarchical summary graph: {hierchical_sum_graph.number_of_nodes()} nodes, "
    f"{hierchical_sum_graph.number_of_edges()} edges"
)
print("Components:", [n for n, d in hierchical_sum_graph.nodes(data=True) if d.get("kind") != "function"])


In [ ]:
# Create and display the interactive Sigma.js view canvas
widget = Sigma(
    hierchical_sum_graph,
    start_layout=True,              
    edge_color="type",
    edge_size="weight",
    default_edge_type="arrow",
    height=800,
    clickable_edges=True
)

display(widget)